In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-win_amd64.whl.metadata (13 kB)
  Using cached rustworkx-0.17.1-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
Using cached qiskit-1.2.4-cp38-abi3-win_amd64.whl (4.6 MB)
Using cached rustworkx-0.17.1-cp39-abi3-win_amd64.whl (2.1 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
   ---------------------------------------- 0.0/17.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.8 MB 330.3 kB/s eta 0:00:54
   ---------------------------------------- 0.1/17.8 MB 1.4 MB/s eta 0:00:13
    --------------------------------------- 0.4/17.8 MB 3.2 MB/s eta 0:00:06
   - -------------------------------------- 0.8/17.8 MB 5.4 MB/s eta 0:00:04
   -- ------------------------------------- 1.1/17.8 MB 5.3 MB/s eta 0:00:04
   --- ------------------------------------ 1.6/17.8 MB 6.5 MB/s eta 0:00:03
   ---- ----------------------------------- 2.1/17.8 MB 7.1 MB/s eta 0:00:03
   --

In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit.circuit.library import UnitaryGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.
# To change the length of the key (along with the number of iterations of the protocol), change the value of N below
N = 1000


#For making the random choice

op = Operator(
    [
        [1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)],
        [math.sqrt(2)/math.sqrt(3), - 1/math.sqrt(3)]
    ]
)

def choose1of3():
  q = QuantumCircuit(2)
  q.h(1)
  q.append(op,[0])
  q.measure_all()
  backend = BasicSimulator()
  compiled = transpile(q,backend)
  results = backend.run(compiled,shots=1).result()
  counts = results.get_counts(compiled)
  bitstring = list(counts.keys())[0]
  q0 = bitstring[1]
  q1 = bitstring[0]

  if q0 == '0': #option 1 (probability 1/3)
    return 1
  elif q0 == '1':
    if q1 == '0': #option 2 (probability 2/3 * 1/2 = 1/3)
      return 2
    elif q1 == '1': #option 3 (probability 2/3 * 1/2 = 1/3)
      return 3
    else:
      print("ERROR. Basis couldn't be selected.")


#Defining the observables X, Z, W and V

X = Operator([[0,1],[1,0]])
Z = Operator([[1,0],[0,-1]])
W = Operator(
    [
        [1/math.sqrt(2), 1/math.sqrt(2)],
        [1/math.sqrt(2), - 1/math.sqrt(2)]
        ]
    )
V = Operator(
    [
        [- 1/math.sqrt(2), 1/math.sqrt(2)],
        [1/math.sqrt(2),   1/math.sqrt(2)]
        ]
    )

#Rotation matrices for measuring X, Z, W and V

constant_a = 1/math.sqrt(4-2*math.sqrt(2))
constant_b = 1/math.sqrt(4+2*math.sqrt(2))
constant_c = math.sqrt(2)-1
constant_d = math.sqrt(2)+1

#X_R is just the Hadamard gate (diagonal basis)
#Z_R is just I, so nothing (standard basis)
W_R = Operator([
    [ constant_a              , constant_b ],
    [ constant_c * constant_a , -constant_d * constant_b ]
])
V_R = Operator([
    [constant_d * constant_b  , -constant_b],
    [-constant_b              , -constant_d * constant_b]
])


#Now, the protocol itself

bob_bases = []
alice_bases = []
bob_measurements = []
alice_measurements = []
final_key = []
totals_and_freqs = { # For calculating |S|
    "11":[0,0],   # X⊗W
    "13":[0,0],   # X⊗V
    "31":[0,0],   # Z⊗W
    "33":[0,0],   # Z⊗V
}

for i in range(9*N//2):

  #Setting up an entangled qubit pair
  q = QuantumCircuit(2,2)
  q.x(0)
  q.h(1)
  q.cx(1,0)
  q.z(1)

  #Qubit 0 is sent to Alice, and qubit 1 is sent to Bob

  #Alice chooses an operator, Bob chooses an operator
  alice_choice = choose1of3()
  bob_choice = choose1of3()

  if alice_choice == 1: # X basis
    q.h(1)
    alice_bases.append(1)
  elif alice_choice == 2: # W basis
    q.append(W_R,[1])
    alice_bases.append(2)
  elif alice_choice == 3: # Z basis
    #no operator - measuring in standard basis
    alice_bases.append(3)

  if bob_choice == 1: # W basis
    q.append(W_R,[0])
    bob_bases.append(1)
  elif bob_choice == 2: # Z basis
    #no operator - measuring in standard basis
    bob_bases.append(2)
  elif bob_choice == 3: # V basis
    q.append(V_R,[0])
    bob_bases.append(3)

  #Measurements
  mapping = {"0":1 , "1":-1}

  #Alice measures operator Ai on her qubit
  q.measure(1,1)
  #Bob measures operator Bj on his qubit
  q.measure(0,0)

  backend = BasicSimulator()
  compiled = transpile(q,backend)
  results = backend.run(compiled,shots=1).result()
  counts = results.get_counts(compiled)
  bitstring = list(counts.keys())[0]
  alice_bit = bitstring[0]
  bob_bit = bitstring[1]

  bob_measurements.append(mapping[bob_bit])
  alice_measurements.append(mapping[alice_bit])


#Now, Alice and Bob share their basis choices, and compare them

for i in range(9*N//2):
  a_basis = alice_bases[i]
  b_basis = bob_bases[i]
  if (a_basis == 2 and b_basis == 1) or (a_basis == 3 and b_basis == 2): # When their basis choices are equal
    final_key.append(alice_measurements[i]) # Using alice's bit, as is pre-agreed between them(bob negates his bit)
  else:
    basis_pair = str(a_basis) + str(b_basis)
    measurement_combined = alice_measurements[i] * bob_measurements[i] # 
    if basis_pair in totals_and_freqs:
      totals_and_freqs[basis_pair][0] += measurement_combined # update total
      totals_and_freqs[basis_pair][1] += 1 # update frequency

#Final key has been generated. Length is N 

#Verifying the security of the communication i.e.: calculating |S|
#First, the average of each pair of basis measurements
average_11 = totals_and_freqs["11"][0]/totals_and_freqs["11"][1]
average_13 = totals_and_freqs["13"][0]/totals_and_freqs["13"][1]
average_31 = totals_and_freqs["31"][0]/totals_and_freqs["31"][1]
average_33 = totals_and_freqs["33"][0]/totals_and_freqs["33"][1]
print("average_11: " + str(average_11))
print("average_13: " + str(average_13))
print("average_31: " + str(average_31))
print("average_33: " + str(average_33))

# S =|⟨X⊗W⟩−⟨X⊗V⟩+⟨Z⊗W⟩+⟨Z⊗V⟩|
S = abs(average_11 - average_13 + average_31 + average_33)

# As no attacker is present, the value of S should be close to 2*sqrt(2) or 2.828
print("S Value: " + str(S))




average_11: -0.7192982456140351
average_13: 0.7172995780590717
average_31: -0.7167019027484144
average_33: -0.7222222222222222
S Value: 2.8755219486437436
